# Notebook 35 — Refresh Supervisor Package

This notebook refreshes the existing supervisor-facing package after the latest pre-feedback additions to the restoration evaluation framework.

It updates the existing supervisor package outputs rather than creating duplicate versioned files.

The refreshed package summarizes:

- the original controlled 50-painting evaluation,
- the refined metric-region policy,
- texture and brushstroke-proxy diagnostics,
- Stable Diffusion uncertainty heatmaps,
- selected per-case diagnostic reports,
- updated Streamlit dashboard assets,
- remaining decisions for supervisor feedback.

The notebook does not rerun models and does not recompute metrics.

In [1]:
from pathlib import Path, PureWindowsPath
import json
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

print("Notebook 35 imports ready.")

Notebook 35 imports ready.


In [2]:
NOTEBOOK_NAME = "35_refresh_supervisor_package_cleaned"

REPO_FOLDER_NAME_CANDIDATES = [
    "painting-restoration-eval",
    "painting_restoration_eval",
]


def find_project_root(
    start_path: Path | None = None,
    repo_folder_name_candidates: list[str] | None = None,
) -> Path:
    if repo_folder_name_candidates is None:
        repo_folder_name_candidates = REPO_FOLDER_NAME_CANDIDATES

    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for parent in [start_path] + list(start_path.parents):
        if parent.name in repo_folder_name_candidates:
            return parent

        if (
            (parent / ".git").exists()
            and (parent / "notebooks").exists()
            and (parent / "data").exists()
            and (parent / "outputs").exists()
        ):
            return parent

        if (
            (parent / "notebooks").exists()
            and (parent / "data").exists()
            and (parent / "outputs").exists()
        ):
            return parent

    raise RuntimeError(
        f"Could not find project root from {start_path}. "
        f"Tried folder names: {repo_folder_name_candidates}"
    )


PROJECT_ROOT = find_project_root()
REPO_FOLDER_NAME = PROJECT_ROOT.name

outputs_dir = PROJECT_ROOT / "outputs"
metrics_dir = outputs_dir / "metrics"
reports_dir = outputs_dir / "reports"
figures_dir = outputs_dir / "figures"
dashboard_dir = outputs_dir / "dashboard"

supervisor_package_dir = outputs_dir / "supervisor_package"
supervisor_package_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Supervisor package directory:", supervisor_package_dir)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Supervisor package directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package


In [3]:
input_paths = {
    # Final refined controlled comparison.
    "refined_comparison": metrics_dir / "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",

    # Texture / brushstroke-proxy diagnostics from Notebook 31.
    "texture_unified": metrics_dir / "comparison_texture_unified_50.csv",
    "texture_case_winners_nonzero": metrics_dir / "comparison_texture_case_winners_nonzero_50.csv",
    "texture_winner_summary_nonzero": metrics_dir / "comparison_texture_winner_summary_nonzero_50.csv",
    "texture_disagreement_cases": metrics_dir / "comparison_texture_disagreement_cases_50.csv",
    "texture_high_texture_brushwork_summary": metrics_dir / "comparison_texture_high_texture_brushwork_summary_50.csv",
    "brushstroke_proxy_summary_by_model": metrics_dir / "comparison_brushstroke_proxy_summary_by_model_50.csv",

    # Stable Diffusion uncertainty heatmap diagnostics from Notebook 32.
    "uncertainty_heatmap_manifest": metrics_dir / "stable_diffusion_uncertainty_heatmap_manifest_50.csv",
    "uncertainty_heatmap_summary_by_case": metrics_dir / "stable_diffusion_uncertainty_heatmap_summary_by_case_50.csv",
    "uncertainty_heatmap_summary_by_mask_type": metrics_dir / "stable_diffusion_uncertainty_heatmap_summary_by_mask_type_50.csv",
    "uncertainty_heatmap_summary_by_category": metrics_dir / "stable_diffusion_uncertainty_heatmap_summary_by_category_50.csv",
    "uncertainty_heatmap_vs_refined_performance": metrics_dir / "stable_diffusion_uncertainty_heatmap_vs_refined_performance_50.csv",
    "uncertainty_heatmap_selected_cases": metrics_dir / "stable_diffusion_uncertainty_heatmap_selected_cases_50.csv",

    # Case diagnostics from Notebook 33.
    "case_diagnostic_selected_cases": metrics_dir / "case_diagnostic_selected_cases_50.csv",
    "case_diagnostic_report_manifest": metrics_dir / "case_diagnostic_report_manifest_50.csv",

    # Dashboard assets from Notebook 34.
    "dashboard_summary": dashboard_dir / "dashboard_summary.json",
    "dashboard_model_winner_summary": dashboard_dir / "dashboard_model_winner_summary.csv",
    "dashboard_metric_vote_summary": dashboard_dir / "dashboard_metric_vote_summary.csv",
    "dashboard_texture_summary": dashboard_dir / "dashboard_texture_summary.csv",
    "dashboard_texture_disagreements": dashboard_dir / "dashboard_texture_disagreements.csv",
    "dashboard_uncertainty_summary": dashboard_dir / "dashboard_uncertainty_summary.csv",
    "dashboard_uncertainty_selected_cases": dashboard_dir / "dashboard_uncertainty_selected_cases.csv",
    "dashboard_case_report_manifest": dashboard_dir / "dashboard_case_report_manifest.csv",
    "dashboard_selected_cases": dashboard_dir / "dashboard_selected_cases.csv",
    "dashboard_figure_manifest": dashboard_dir / "dashboard_figure_manifest.csv",
    "dashboard_asset_manifest": dashboard_dir / "dashboard_asset_manifest.json",

    # HTML reports.
    "uncertainty_heatmap_report": reports_dir / "stable_diffusion_uncertainty_heatmap_report_50.html",
    "case_report_index": reports_dir / "case_diagnostics" / "case_report_index.html",
}

output_paths = {
    # Existing supervisor package outputs are overwritten/refreshed.
    "supervisor_summary": supervisor_package_dir / "supervisor_summary.md",
    "supervisor_artifact_index": supervisor_package_dir / "supervisor_artifact_index.csv",
    "supervisor_key_findings": supervisor_package_dir / "supervisor_key_findings.json",
    "supervisor_open_questions": supervisor_package_dir / "supervisor_open_questions.md",
    "supervisor_feedback_agenda": supervisor_package_dir / "supervisor_feedback_agenda.md",
    "supervisor_package_manifest": supervisor_package_dir / "supervisor_package_manifest.json",
}

print("Input paths:")
for name, path in input_paths.items():
    status = "exists" if path.exists() else "missing"
    print(f"- {name}: {path} [{status}]")

print("\nOutput paths to refresh:")
for name, path in output_paths.items():
    exists_note = "will overwrite existing file" if path.exists() else "will create"
    print(f"- {name}: {path} [{exists_note}]")

Input paths:
- refined_comparison: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_unified_refined_opencv_lama_stable_diffusion_50.csv [exists]
- texture_unified: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_unified_50.csv [exists]
- texture_case_winners_nonzero: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_case_winners_nonzero_50.csv [exists]
- texture_winner_summary_nonzero: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_winner_summary_nonzero_50.csv [exists]
- texture_disagreement_cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_disagreement_cases_50.csv [exists]
- texture_high_texture_brushwork_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_high_texture_brushwork_summary_50.csv [exists]
- brushstroke_proxy_summary_by_model: D:\Masters\FH\Thesis\painting-restoration-e

In [4]:
def get_repo_folder_candidates() -> list[str]:
    candidates = [
        "painting-restoration-eval",
        "painting_restoration_eval",
        REPO_FOLDER_NAME,
    ]

    return list(dict.fromkeys(candidates))


def to_project_relative_path(path_value: str | Path | None) -> str | None:
    if path_value is None:
        return None

    try:
        if pd.isna(path_value):
            return None
    except TypeError:
        pass

    path_text = str(path_value).strip()

    if not path_text:
        return None

    normalized_text = path_text.replace("\\", "/")

    for repo_folder_name in get_repo_folder_candidates():
        marker = f"/{repo_folder_name}/"

        if marker in normalized_text:
            return normalized_text.split(marker, 1)[1]

        if normalized_text.startswith(f"{repo_folder_name}/"):
            return normalized_text.split(f"{repo_folder_name}/", 1)[1]

    path = Path(normalized_text)

    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        pass

    windows_path = PureWindowsPath(path_text)

    for repo_folder_name in get_repo_folder_candidates():
        if repo_folder_name in windows_path.parts:
            repo_index = windows_path.parts.index(repo_folder_name)
            relative_parts = windows_path.parts[repo_index + 1:]
            return Path(*relative_parts).as_posix()

    return normalized_text


def resolve_project_path(path_value: str | Path | None) -> Path | None:
    if path_value is None:
        return None

    try:
        if pd.isna(path_value):
            return None
    except TypeError:
        pass

    path_text = str(path_value).strip()

    if not path_text:
        return None

    normalized_text = path_text.replace("\\", "/")

    for repo_folder_name in get_repo_folder_candidates():
        marker = f"/{repo_folder_name}/"

        if marker in normalized_text:
            relative_part = normalized_text.split(marker, 1)[1]
            return PROJECT_ROOT / relative_part

        if normalized_text.startswith(f"{repo_folder_name}/"):
            relative_part = normalized_text.split(f"{repo_folder_name}/", 1)[1]
            return PROJECT_ROOT / relative_part

    windows_path = PureWindowsPath(path_text)

    for repo_folder_name in get_repo_folder_candidates():
        if repo_folder_name in windows_path.parts:
            repo_index = windows_path.parts.index(repo_folder_name)
            relative_parts = windows_path.parts[repo_index + 1:]
            return PROJECT_ROOT.joinpath(*relative_parts)

    path = Path(path_text)

    if path.is_absolute():
        if path.exists():
            return path

        for repo_folder_name in get_repo_folder_candidates():
            if repo_folder_name in path.parts:
                repo_index = path.parts.index(repo_folder_name)
                relative_parts = path.parts[repo_index + 1:]
                return PROJECT_ROOT.joinpath(*relative_parts)

        return path

    return PROJECT_ROOT / path


def file_exists_from_value(path_value: str | Path | None) -> bool:
    resolved_path = resolve_project_path(path_value)
    return bool(resolved_path is not None and resolved_path.exists())


def load_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    return pd.read_csv(path)


def load_json_if_exists(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}

    return json.loads(path.read_text(encoding="utf-8"))


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def find_first_existing_column(
    df: pd.DataFrame,
    candidates: list[str],
) -> str | None:
    for column in candidates:
        if column in df.columns:
            return column

    return None


def safe_int(value: Any, default: int = 0) -> int:
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass

    try:
        return int(value)
    except Exception:
        return default


def safe_float(value: Any, default: float = np.nan) -> float:
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass

    try:
        return float(value)
    except Exception:
        return default


print("Notebook 35 helpers ready.")

Notebook 35 helpers ready.


In [5]:
source_dataframes = {}
source_json = {}

for key, path in input_paths.items():
    if path.suffix.lower() == ".csv":
        source_dataframes[key] = load_csv_if_exists(path)
        print(f"CSV {key}: {source_dataframes[key].shape}")
    elif path.suffix.lower() == ".json":
        source_json[key] = load_json_if_exists(path)
        print(f"JSON {key}: {'loaded' if source_json[key] else 'empty/missing'}")
    else:
        print(f"File {key}: {'exists' if path.exists() else 'missing'}")

refined_comparison_df = source_dataframes["refined_comparison"]
texture_winner_summary_df = source_dataframes["texture_winner_summary_nonzero"]
texture_disagreement_df = source_dataframes["texture_disagreement_cases"]
brushstroke_summary_df = source_dataframes["brushstroke_proxy_summary_by_model"]
uncertainty_case_df = source_dataframes["uncertainty_heatmap_summary_by_case"]
uncertainty_mask_df = source_dataframes["uncertainty_heatmap_summary_by_mask_type"]
uncertainty_category_df = source_dataframes["uncertainty_heatmap_summary_by_category"]
case_selected_df = source_dataframes["case_diagnostic_selected_cases"]
case_manifest_df = source_dataframes["case_diagnostic_report_manifest"]

dashboard_summary = source_json.get("dashboard_summary", {})
dashboard_asset_manifest = source_json.get("dashboard_asset_manifest", {})

print("\nCore dataframe shapes:")
print("refined_comparison_df:", refined_comparison_df.shape)
print("texture_winner_summary_df:", texture_winner_summary_df.shape)
print("texture_disagreement_df:", texture_disagreement_df.shape)
print("brushstroke_summary_df:", brushstroke_summary_df.shape)
print("uncertainty_case_df:", uncertainty_case_df.shape)
print("uncertainty_mask_df:", uncertainty_mask_df.shape)
print("uncertainty_category_df:", uncertainty_category_df.shape)
print("case_selected_df:", case_selected_df.shape)
print("case_manifest_df:", case_manifest_df.shape)

CSV refined_comparison: (200, 40)
CSV texture_unified: (750, 102)
CSV texture_case_winners_nonzero: (200, 16)
CSV texture_winner_summary_nonzero: (3, 2)
CSV texture_disagreement_cases: (200, 21)
CSV texture_high_texture_brushwork_summary: (15, 11)
CSV brushstroke_proxy_summary_by_model: (3, 6)
CSV uncertainty_heatmap_manifest: (40, 11)
CSV uncertainty_heatmap_summary_by_case: (40, 60)
CSV uncertainty_heatmap_summary_by_mask_type: (4, 10)
CSV uncertainty_heatmap_summary_by_category: (5, 10)
CSV uncertainty_heatmap_vs_refined_performance: (40, 119)
CSV uncertainty_heatmap_selected_cases: (22, 124)
CSV case_diagnostic_selected_cases: (30, 98)
CSV case_diagnostic_report_manifest: (30, 24)
JSON dashboard_summary: loaded
CSV dashboard_model_winner_summary: (4, 3)
CSV dashboard_metric_vote_summary: (3, 5)
CSV dashboard_texture_summary: (21, 15)
CSV dashboard_texture_disagreements: (200, 6)
CSV dashboard_uncertainty_summary: (9, 12)
CSV dashboard_uncertainty_selected_cases: (22, 12)
CSV dashbo

In [6]:
required_input_keys = [
    "refined_comparison",
    "case_diagnostic_selected_cases",
    "case_diagnostic_report_manifest",
    "dashboard_summary",
    "dashboard_asset_manifest",
    "case_report_index",
]

missing_required_inputs = [
    key
    for key in required_input_keys
    if not input_paths[key].exists()
]

if missing_required_inputs:
    for key in missing_required_inputs:
        print(f"Missing required input: {key} -> {input_paths[key]}")

    raise FileNotFoundError("One or more required supervisor package inputs are missing.")

if refined_comparison_df.empty:
    raise ValueError("Refined comparison dataframe is empty.")

if refined_comparison_df["case_id"].nunique() != 200:
    raise ValueError(
        f"Expected 200 non-zero refined comparison cases, got {refined_comparison_df['case_id'].nunique()}."
    )

if "zero_control" in set(refined_comparison_df["mask_type"].astype(str)):
    raise ValueError("Refined comparison unexpectedly contains zero_control cases.")

if case_selected_df.empty:
    raise ValueError("Case diagnostic selected cases dataframe is empty.")

if case_manifest_df.empty:
    raise ValueError("Case diagnostic report manifest dataframe is empty.")

if not input_paths["case_report_index"].exists():
    raise FileNotFoundError("Case diagnostic report index is missing.")

if not dashboard_summary:
    raise ValueError("Dashboard summary JSON is empty or missing.")

if not dashboard_asset_manifest:
    raise ValueError("Dashboard asset manifest JSON is empty or missing.")

optional_but_expected_inputs = [
    "texture_winner_summary_nonzero",
    "texture_disagreement_cases",
    "brushstroke_proxy_summary_by_model",
    "uncertainty_heatmap_summary_by_case",
    "uncertainty_heatmap_summary_by_mask_type",
    "uncertainty_heatmap_summary_by_category",
    "uncertainty_heatmap_report",
]

missing_optional_but_expected = [
    key
    for key in optional_but_expected_inputs
    if not input_paths[key].exists()
]

if missing_optional_but_expected:
    print("Missing optional-but-expected inputs:")
    for key in missing_optional_but_expected:
        print(f"- {key}: {input_paths[key]}")

print("Supervisor package input validation passed.")
print("Non-zero refined cases:", refined_comparison_df["case_id"].nunique())
print("Selected case reports:", case_selected_df["case_id"].nunique())
print("Dashboard asset entries:", len(dashboard_asset_manifest.get("assets", {})))

Supervisor package input validation passed.
Non-zero refined cases: 200
Selected case reports: 30
Dashboard asset entries: 11


In [7]:
winner_column = find_first_existing_column(
    refined_comparison_df,
    [
        "overall_metric_vote",
        "refined_overall_metric_vote",
        "refined_metric_winner",
        "majority_vote_winner",
        "winner",
    ],
)

if winner_column is None:
    print("Available refined comparison columns:")
    display(pd.DataFrame({"column": refined_comparison_df.columns}))
    raise ValueError("Could not find refined winner column.")

winner_summary_df = (
    refined_comparison_df[winner_column]
    .fillna("missing")
    .value_counts()
    .rename_axis("refined_winner")
    .reset_index(name="case_count")
)

winner_summary_df["case_share"] = (
    winner_summary_df["case_count"] / winner_summary_df["case_count"].sum()
)

metric_vote_columns = {
    "OpenCV Telea": find_first_existing_column(
        refined_comparison_df,
        ["refined_opencv_metric_wins", "opencv_telea_metric_wins", "opencv_metric_wins"],
    ),
    "LaMa": find_first_existing_column(
        refined_comparison_df,
        ["refined_lama_metric_wins", "lama_metric_wins"],
    ),
    "Stable Diffusion Inpainting": find_first_existing_column(
        refined_comparison_df,
        [
            "refined_stable_diffusion_metric_wins",
            "stable_diffusion_inpainting_metric_wins",
            "stable_diffusion_metric_wins",
        ],
    ),
}

metric_vote_summary = {}

for model_name, column in metric_vote_columns.items():
    if column is None:
        metric_vote_summary[model_name] = {
            "metric_vote_column": None,
            "total_metric_votes": None,
            "mean_metric_votes": None,
        }
    else:
        metric_vote_summary[model_name] = {
            "metric_vote_column": column,
            "total_metric_votes": float(refined_comparison_df[column].sum()),
            "mean_metric_votes": float(refined_comparison_df[column].mean()),
        }

category_counts = (
    refined_comparison_df["category"]
    .astype(str)
    .value_counts()
    .sort_index()
    .to_dict()
)

mask_type_counts = (
    refined_comparison_df["mask_type"]
    .astype(str)
    .value_counts()
    .sort_index()
    .to_dict()
)

uncertainty_case_count = int(uncertainty_case_df["case_id"].nunique()) if not uncertainty_case_df.empty and "case_id" in uncertainty_case_df.columns else 0

uncertainty_seed_output_count = 0

if not source_dataframes["uncertainty_heatmap_manifest"].empty:
    manifest_df = source_dataframes["uncertainty_heatmap_manifest"]

    if "uncertainty_generation_id" in manifest_df.columns:
        uncertainty_seed_output_count = int(manifest_df["uncertainty_generation_id"].nunique())
    else:
        uncertainty_seed_output_count = int(len(manifest_df))

selected_case_count = int(case_selected_df["case_id"].nunique())

selected_cases_with_uncertainty = (
    int(case_selected_df["has_uncertainty_heatmap"].fillna(False).astype(bool).sum())
    if "has_uncertainty_heatmap" in case_selected_df.columns
    else 0
)

selected_cases_with_texture_disagreement = (
    int(case_selected_df["has_texture_disagreement"].fillna(False).astype(bool).sum())
    if "has_texture_disagreement" in case_selected_df.columns
    else 0
)

texture_disagreement_count = (
    int(texture_disagreement_df["case_id"].nunique())
    if not texture_disagreement_df.empty and "case_id" in texture_disagreement_df.columns
    else 0
)

dashboard_assets_count = len(dashboard_asset_manifest.get("assets", {}))

supervisor_facts = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": NOTEBOOK_NAME,
    "dataset": {
        "controlled_paintings": 50,
        "non_zero_cases": int(refined_comparison_df["case_id"].nunique()),
        "categories": sorted(refined_comparison_df["category"].astype(str).unique().tolist()),
        "non_zero_mask_types": sorted(refined_comparison_df["mask_type"].astype(str).unique().tolist()),
        "category_counts": category_counts,
        "mask_type_counts": mask_type_counts,
    },
    "models": {
        "fully_evaluated": [
            "OpenCV Telea",
            "LaMa",
            "Stable Diffusion Inpainting",
        ],
        "feasibility_audited_only": [
            "SDXL Inpainting",
        ],
    },
    "refined_comparison": {
        "winner_column": winner_column,
        "winner_summary": winner_summary_df.to_dict(orient="records"),
        "metric_vote_summary": metric_vote_summary,
    },
    "new_pre_feedback_additions": {
        "texture_brushstroke_proxy_notebook": "31_texture_metrics_cleaned.ipynb",
        "stable_diffusion_uncertainty_heatmaps_notebook": "32_uncertainty_heatmaps_cleaned.ipynb",
        "case_diagnostic_reports_notebook": "33_case_report_generation_cleaned.ipynb",
        "dashboard_assets_notebook": "34_prepare_final_dashboard_assets_cleaned.ipynb",
        "updated_streamlit_app": "streamlit_app.py",
    },
    "texture_diagnostics": {
        "texture_disagreement_cases": texture_disagreement_count,
        "brushstroke_proxy_summary_available": not brushstroke_summary_df.empty,
    },
    "uncertainty_heatmaps": {
        "cases": uncertainty_case_count,
        "seed_outputs": uncertainty_seed_output_count,
        "seeds_per_case_expected": 4,
        "interpretation": "Seed-based spatial variability, not calibrated confidence.",
    },
    "case_reports": {
        "selected_cases": selected_case_count,
        "selected_cases_with_uncertainty_heatmaps": selected_cases_with_uncertainty,
        "selected_cases_with_texture_disagreement": selected_cases_with_texture_disagreement,
        "case_report_index": to_project_relative_path(input_paths["case_report_index"]),
    },
    "dashboard": {
        "dashboard_assets_count": dashboard_assets_count,
        "streamlit_app": "streamlit_app.py",
        "dashboard_assets_dir": to_project_relative_path(dashboard_dir),
    },
}

print("Supervisor facts prepared.")
print(json.dumps(supervisor_facts, indent=2, ensure_ascii=False))

display(winner_summary_df)

Supervisor facts prepared.
{
  "generated_at": "2026-07-07T22:29:38",
  "notebook": "35_refresh_supervisor_package_cleaned",
  "dataset": {
    "controlled_paintings": 50,
    "non_zero_cases": 200,
    "categories": [
      "abstraction_surrealism",
      "architecture_structured",
      "high_texture_brushwork",
      "landscape_natural",
      "portrait_figure"
    ],
    "non_zero_mask_types": [
      "loss_large",
      "loss_small",
      "mixed_damage",
      "scratch_thin"
    ],
    "category_counts": {
      "abstraction_surrealism": 40,
      "architecture_structured": 40,
      "high_texture_brushwork": 40,
      "landscape_natural": 40,
      "portrait_figure": 40
    },
    "mask_type_counts": {
      "loss_large": 50,
      "loss_small": 50,
      "mixed_damage": 50,
      "scratch_thin": 50
    }
  },
  "models": {
    "fully_evaluated": [
      "OpenCV Telea",
      "LaMa",
      "Stable Diffusion Inpainting"
    ],
    "feasibility_audited_only": [
      "SDXL Inpaint

,refined_winner,case_count,case_share
0,lama,155,0.775
1,tie_lama_opencv_telea,23,0.115
2,opencv_telea,21,0.105
3,stable_diffusion_inpainting,1,0.005


In [9]:
winner_lines = []

for row in supervisor_facts["refined_comparison"]["winner_summary"]:
    winner_lines.append(
        f"- {row['refined_winner']}: {row['case_count']} cases "
        f"({row['case_share']:.1%})"
    )

winner_summary_text = "\n".join(winner_lines)

category_lines = "\n".join(
    f"- {category}: {count} cases"
    for category, count in supervisor_facts["dataset"]["category_counts"].items()
)

mask_lines = "\n".join(
    f"- {mask_type}: {count} cases"
    for mask_type, count in supervisor_facts["dataset"]["mask_type_counts"].items()
)

supervisor_summary_md = f"""
# Supervisor Summary — Pre-Feedback Package

Generated by `{NOTEBOOK_NAME}` on {supervisor_facts["generated_at"]}.

## Thesis topic

**Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration**

Core claim:

> Visual plausibility is not the same as restoration trustworthiness.

The current package presents a controlled evaluation framework for AI-assisted painting restoration. The work focuses on evaluation, diagnostics, and trustworthiness rather than training a new restoration model.

## Current controlled experiment status

The current framework evaluates a controlled 50-painting subset across synthetic damage cases.

- Controlled paintings: **50**
- Main comparison cases: **200 non-zero damage cases**
- Zero-control cases are excluded from the refined model comparison
- Fully evaluated models:
  - OpenCV Telea
  - LaMa
  - Stable Diffusion Inpainting
- Feasibility-audited only:
  - SDXL Inpainting

## Dataset coverage

### Painting categories

{category_lines}

### Non-zero mask types

{mask_lines}

## Refined model comparison

The refined comparison uses the final metric-region policy:

- MSE: masked region
- PSNR: masked region
- SSIM: mask bounding-box crop
- LPIPS: mask bounding-box crop
- CLIP: mask bounding-box crop
- DINOv2: mask bounding-box crop

Refined winner column used: `{winner_column}`.

### Refined winner summary

{winner_summary_text}

The main reference-based result remains that LaMa is strongest under the refined metric policy. OpenCV Telea remains a useful deterministic baseline. Stable Diffusion rarely wins under reference-based metrics, but remains important for studying generative plausibility and instability.

## What was added since the original supervisor package

The original supervisor package covered the controlled subset, model stack, refined metric policy, and final model comparison.

The refreshed package adds the following layers.

### 1. Texture and brushstroke-proxy diagnostics

Notebook:

- `31_texture_metrics_cleaned.ipynb`

Main additions:

- texture-aware metric layer,
- GLCM and Gabor texture descriptors,
- brushstroke-proxy directional texture descriptors,
- texture winner summaries,
- texture/refined disagreement cases,
- high-texture brushwork focus.

Important interpretation boundary:

> These are brushstroke-proxy diagnostics, not semantic brushstroke recognition, authentication, or conservation judgment.

### 2. Stable Diffusion uncertainty heatmaps

Notebook:

- `32_uncertainty_heatmaps_cleaned.ipynb`

Main additions:

- seed-based spatial uncertainty maps,
- masked-region uncertainty summaries,
- bounding-box uncertainty summaries,
- outside-mask uncertainty summaries,
- outside boundary-ring uncertainty summaries,
- uncertainty summaries by mask type and category,
- uncertainty versus refined reference-performance table.

Current uncertainty subset:

- Cases: **{uncertainty_case_count}**
- Seed outputs: **{uncertainty_seed_output_count}**
- Expected seeds per case: **4**

Important interpretation boundary:

> This is seed-based spatial variability, not calibrated model confidence.

### 3. Per-case diagnostic reports

Notebook:

- `33_case_report_generation_cleaned.ipynb`

Main additions:

- selected case diagnostic grids,
- individual HTML reports,
- case report index,
- combined evidence from visual outputs, refined metrics, texture diagnostics, and uncertainty diagnostics.

Current selected case reports:

- Selected cases: **{selected_case_count}**
- Selected cases with uncertainty heatmaps: **{selected_cases_with_uncertainty}**
- Selected cases with texture/refined disagreement flag: **{selected_cases_with_texture_disagreement}**

Main entry point:

- `{to_project_relative_path(input_paths["case_report_index"])}`

### 4. Final dashboard assets and updated Streamlit app

Notebook:

- `34_prepare_final_dashboard_assets_cleaned.ipynb`

Updated app:

- `streamlit_app.py`

Main additions:

- dashboard-ready CSV/JSON assets,
- Texture Diagnostics page,
- Case Reports page,
- updated Diffusion Uncertainty page,
- updated Reports and Debug pages.

Dashboard source directory:

- `{to_project_relative_path(dashboard_dir)}`

Run command:

    streamlit run streamlit_app.py

## Current supervisor-review package

The project is now ready for supervisor feedback on:

1. Whether the 50-painting controlled subset is sufficient for the thesis scope.
2. Whether the refined metric-region policy is acceptable.
3. Whether texture and brushstroke-proxy diagnostics should remain part of the core framework.
4. Whether the 40-case Stable Diffusion uncertainty heatmap subset is sufficient before scaling.
5. Whether SDXL should remain feasibility-audited only.
6. Whether the Streamlit dashboard should be treated as a formal supporting artifact.
7. Which optional extensions should be pursued after feedback.

## Recommended framing for feedback meeting

The current framework should be presented as a completed pre-feedback evaluation prototype.

The most important supervisor decision is not which model wins. The key decision is whether the framework layers are methodologically acceptable:

- reference-based metrics,
- region-aware metric policy,
- perceptual and feature similarity,
- texture and brushstroke-proxy diagnostics,
- seed-based uncertainty maps,
- per-case inspection reports,
- dashboard-based review interface.

## Remaining work after supervisor feedback

The following items should wait until after supervisor feedback:

1. Semantic or iconographic consistency checks.
2. Metadata-driven analysis by artist, medium, period, collection, or other artwork metadata.
3. Metric-policy ablation.
4. Scaling beyond the controlled 50-painting subset.
5. Extending uncertainty heatmaps beyond the current 40-case subset.
6. SDXL follow-up if better hardware or supervisor priority justifies it.
7. Possible human/expert review protocol.
8. Thesis chapter structuring and paper-style condensation.

These are intentionally not started before feedback to avoid uncontrolled scope expansion.
"""

write_text(
    output_paths["supervisor_summary"],
    supervisor_summary_md,
)

print("Saved:", output_paths["supervisor_summary"])
print(output_paths["supervisor_summary"].read_text(encoding="utf-8")[:3000])

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_summary.md
# Supervisor Summary — Pre-Feedback Package

Generated by `35_refresh_supervisor_package_cleaned` on 2026-07-07T22:29:38.

## Thesis topic

**Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration**

Core claim:

> Visual plausibility is not the same as restoration trustworthiness.

The current package presents a controlled evaluation framework for AI-assisted painting restoration. The work focuses on evaluation, diagnostics, and trustworthiness rather than training a new restoration model.

## Current controlled experiment status

The current framework evaluates a controlled 50-painting subset across synthetic damage cases.

- Controlled paintings: **50**
- Main comparison cases: **200 non-zero damage cases**
- Zero-control cases are excluded from the refined model comparison
- Fully evaluated models:
  - OpenCV Telea
  - LaMa
  - Stable Diffusion Inpainting
- Feasibi

In [10]:
supervisor_key_findings = {
    "generated_by": NOTEBOOK_NAME,
    "generated_at": supervisor_facts["generated_at"],
    "findings": [
        {
            "finding_id": "F1",
            "title": "Controlled evaluation framework is complete for the pre-feedback checkpoint",
            "summary": (
                "The current framework covers 50 paintings, 200 non-zero restoration cases, "
                "three fully evaluated restoration models, and one feasibility-audited model."
            ),
            "evidence": [
                "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
                "dashboard_summary.json",
            ],
            "supervisor_decision_needed": (
                "Confirm whether this controlled subset is sufficient for the thesis scope before scaling."
            ),
        },
        {
            "finding_id": "F2",
            "title": "LaMa remains strongest under the refined reference-based metric policy",
            "summary": (
                "The refined comparison shows LaMa as the dominant model under the selected full-reference "
                "metrics and region policy."
            ),
            "evidence": [
                "dashboard_model_winner_summary.csv",
                "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
            ],
            "supervisor_decision_needed": (
                "Confirm whether the majority-vote summary is an acceptable compact model-comparison result."
            ),
        },
        {
            "finding_id": "F3",
            "title": "Metric-region policy is a central methodological contribution",
            "summary": (
                "The framework distinguishes masked-region metrics from mask-bounding-box metrics. "
                "This prevents structurally inappropriate use of metrics such as SSIM on sparse masked pixels."
            ),
            "evidence": [
                "26_refined_metric_region_policy_cleaned.ipynb",
                "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
            ],
            "supervisor_decision_needed": (
                "Confirm whether the final metric-region policy should remain fixed for thesis experiments."
            ),
        },
        {
            "finding_id": "F4",
            "title": "Texture and brushstroke-proxy diagnostics add a local-structure layer",
            "summary": (
                "Notebook 31 adds texture and directional structure diagnostics, including GLCM, Gabor, "
                "and brushstroke-proxy features."
            ),
            "evidence": [
                "comparison_texture_unified_50.csv",
                "comparison_texture_case_winners_nonzero_50.csv",
                "comparison_texture_disagreement_cases_50.csv",
                "comparison_brushstroke_proxy_summary_by_model_50.csv",
            ],
            "supervisor_decision_needed": (
                "Confirm whether these should be part of the core framework or treated as supplementary diagnostics."
            ),
            "interpretation_boundary": (
                "Brushstroke-proxy metrics are not semantic brushstroke recognition or authentication."
            ),
        },
        {
            "finding_id": "F5",
            "title": "Stable Diffusion uncertainty heatmaps expose spatial instability",
            "summary": (
                "Notebook 32 converts multi-seed Stable Diffusion variation into spatial uncertainty heatmaps. "
                "This reveals where the generative model is unstable within and around damaged regions."
            ),
            "evidence": [
                "stable_diffusion_uncertainty_heatmap_summary_by_case_50.csv",
                "stable_diffusion_uncertainty_heatmap_summary_by_mask_type_50.csv",
                "stable_diffusion_uncertainty_heatmap_report_50.html",
            ],
            "supervisor_decision_needed": (
                "Confirm whether the current 40-case subset is sufficient or should be expanded after feedback."
            ),
            "interpretation_boundary": (
                "The heatmaps show seed-based spatial variability, not calibrated confidence."
            ),
        },
        {
            "finding_id": "F6",
            "title": "Case reports make aggregate findings inspectable",
            "summary": (
                "Notebook 33 creates selected per-case diagnostic reports combining visual outputs, refined metrics, "
                "texture diagnostics, and uncertainty heatmap evidence."
            ),
            "evidence": [
                "case_diagnostic_selected_cases_50.csv",
                "case_diagnostic_report_manifest_50.csv",
                "case_report_index.html",
            ],
            "supervisor_decision_needed": (
                "Confirm whether selected case reports should be used as thesis evidence examples."
            ),
        },
        {
            "finding_id": "F7",
            "title": "The Streamlit dashboard is now a review interface for the framework",
            "summary": (
                "Notebook 34 prepares dashboard assets and the updated Streamlit app exposes model comparison, "
                "texture diagnostics, uncertainty diagnostics, and selected case reports."
            ),
            "evidence": [
                "dashboard_asset_manifest.json",
                "streamlit_app.py",
            ],
            "supervisor_decision_needed": (
                "Confirm whether the dashboard should remain a supporting artifact for the thesis."
            ),
        },
        {
            "finding_id": "F8",
            "title": "Scope expansion should wait until after supervisor feedback",
            "summary": (
                "Potential extensions include semantic/iconographic consistency checks, metadata-driven analysis, "
                "metric-policy ablation, scaling beyond 50 paintings, full uncertainty expansion, SDXL follow-up, "
                "and human/expert review."
            ),
            "evidence": [
                "supervisor_open_questions.md",
                "supervisor_feedback_agenda.md",
            ],
            "supervisor_decision_needed": (
                "Prioritize which extensions, if any, should be pursued after feedback."
            ),
        },
    ],
}

write_json(
    output_paths["supervisor_key_findings"],
    supervisor_key_findings,
)

print("Saved:", output_paths["supervisor_key_findings"])
print(json.dumps(supervisor_key_findings, indent=2, ensure_ascii=False)[:4000])

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_key_findings.json
{
  "generated_by": "35_refresh_supervisor_package_cleaned",
  "generated_at": "2026-07-07T22:29:38",
  "findings": [
    {
      "finding_id": "F1",
      "title": "Controlled evaluation framework is complete for the pre-feedback checkpoint",
      "summary": "The current framework covers 50 paintings, 200 non-zero restoration cases, three fully evaluated restoration models, and one feasibility-audited model.",
      "evidence": [
        "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
        "dashboard_summary.json"
      ],
      "supervisor_decision_needed": "Confirm whether this controlled subset is sufficient for the thesis scope before scaling."
    },
    {
      "finding_id": "F2",
      "title": "LaMa remains strongest under the refined reference-based metric policy",
      "summary": "The refined comparison shows LaMa as the dominant model under th

In [11]:
supervisor_open_questions_md = """
# Supervisor Open Questions

This file lists the main decisions to confirm before expanding the thesis work further.

## 1. Controlled subset size

Current state:

- 50 paintings
- 5 categories
- 200 non-zero restoration cases
- 3 fully evaluated models
- SDXL feasibility-audited only

Question:

> Is the controlled 50-painting subset sufficient for the thesis, or should the framework be scaled after feedback?

Recommended pre-feedback position:

> Treat the 50-painting subset as sufficient for the methodological prototype. Scale only if the supervisor explicitly requests stronger empirical coverage.

## 2. Metric-region policy

Current final policy:

- MSE: masked region
- PSNR: masked region
- SSIM: mask bounding-box crop
- LPIPS: mask bounding-box crop
- CLIP: mask bounding-box crop
- DINOv2: mask bounding-box crop
- Texture metrics: mask bounding-box crop
- Brushstroke-proxy metrics: mask bounding-box crop

Question:

> Is this metric-region policy acceptable as the fixed policy for the thesis?

Recommended pre-feedback position:

> Keep this policy fixed unless the supervisor asks for ablation or comparison.

## 3. Texture and brushstroke-proxy diagnostics

Current state:

- Texture metrics added in Notebook 31.
- Brushstroke-proxy orientation and directional texture diagnostics added.
- Disagreement cases between refined metrics and texture diagnostics are available.

Question:

> Should texture and brushstroke-proxy diagnostics be part of the core framework or treated as supplementary diagnostics?

Recommended pre-feedback position:

> Keep them as core diagnostic layers, but phrase them conservatively.

Important wording:

> Brushstroke-proxy metrics are directional texture proxies, not semantic brushstroke recognition, authentication, or conservation truth.

## 4. Stable Diffusion uncertainty heatmaps

Current state:

- 40 cases.
- 4 seeds per case.
- 160 seed outputs.
- Spatial heatmaps generated from per-pixel variation across seed outputs.
- Summaries available by case, mask type, and category.

Question:

> Is the current 40-case uncertainty heatmap subset sufficient, or should uncertainty be expanded to all 200 non-zero cases?

Recommended pre-feedback position:

> Keep the current 40-case uncertainty subset for feedback. Expand only after supervisor approval because full expansion increases compute and storage.

Important wording:

> These heatmaps show seed-based spatial variability, not calibrated confidence.

## 5. SDXL follow-up

Current state:

- SDXL was feasibility-audited.
- Full local evaluation was not completed because of local GPU/runtime constraints.

Question:

> Should SDXL remain feasibility-audited only, or should it be retried with better hardware or cloud resources?

Recommended pre-feedback position:

> Keep SDXL as feasibility-audited only unless the supervisor considers it essential.

## 6. Semantic or iconographic consistency checks

Possible future extension:

- CLIP prompt consistency.
- Region-level description comparison.
- Manual annotation of semantic preservation.
- Iconographic mismatch analysis.

Question:

> Should the thesis include semantic/iconographic preservation checks, or would that expand the scope too far?

Recommended pre-feedback position:

> Wait. This can become subjective quickly and should be supervisor-approved before implementation.

## 7. Metadata-driven analysis

Possible future extension:

- analyze results by artist,
- medium,
- source collection,
- artwork period,
- creation century,
- genre/category.

Question:

> Should artwork metadata be used as an analysis dimension?

Recommended pre-feedback position:

> Only add this if metadata is clean enough and the supervisor wants a stronger art-historical framing.

## 8. Metric-policy ablation

Possible future extension:

- old vs refined metric-region policy,
- reference metrics only vs perceptual/feature metrics,
- with vs without texture diagnostics,
- with vs without uncertainty diagnostics,
- alternative majority vote rules.

Question:

> Should the thesis include an ablation showing how evaluation conclusions change under different policy choices?

Recommended pre-feedback position:

> This is methodologically valuable, but should wait until the supervisor confirms the current framework.

## 9. Human or expert review

Possible future extension:

- small human visual preference study,
- expert conservator review if available,
- structured rubric comparing plausibility, fidelity, and risk.

Question:

> Is a human/expert review expected, useful, or out of scope?

Recommended pre-feedback position:

> Treat it as optional. Do not start before feedback.

## 10. Dashboard role

Current state:

- Streamlit dashboard updated.
- Dashboard includes overview, model comparison, texture diagnostics, uncertainty heatmaps, case reports, key findings, reports, and debug pages.

Question:

> Should the dashboard be treated as a formal supporting artifact in the thesis submission?

Recommended pre-feedback position:

> Yes, as a supporting artifact and inspection tool, not as the primary research result.
"""

write_text(
    output_paths["supervisor_open_questions"],
    supervisor_open_questions_md,
)

print("Saved:", output_paths["supervisor_open_questions"])
print(output_paths["supervisor_open_questions"].read_text(encoding="utf-8")[:3000])

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_open_questions.md
# Supervisor Open Questions

This file lists the main decisions to confirm before expanding the thesis work further.

## 1. Controlled subset size

Current state:

- 50 paintings
- 5 categories
- 200 non-zero restoration cases
- 3 fully evaluated models
- SDXL feasibility-audited only

Question:

> Is the controlled 50-painting subset sufficient for the thesis, or should the framework be scaled after feedback?

Recommended pre-feedback position:

> Treat the 50-painting subset as sufficient for the methodological prototype. Scale only if the supervisor explicitly requests stronger empirical coverage.

## 2. Metric-region policy

Current final policy:

- MSE: masked region
- PSNR: masked region
- SSIM: mask bounding-box crop
- LPIPS: mask bounding-box crop
- CLIP: mask bounding-box crop
- DINOv2: mask bounding-box crop
- Texture metrics: mask bounding-box crop
- Brushstroke-prox

In [12]:
supervisor_feedback_agenda_md = f"""
# Supervisor Feedback Agenda

This agenda is intended for the next supervisor meeting.

## 1. One-sentence project status

The project now has a working pre-feedback evaluation framework for AI-assisted painting restoration, including refined full-reference metrics, texture and brushstroke-proxy diagnostics, Stable Diffusion uncertainty heatmaps, selected per-case diagnostic reports, and an updated Streamlit dashboard.

## 2. Main artifacts to show

### A. Streamlit dashboard

Run command:

    streamlit run streamlit_app.py

Recommended pages to show:

1. Overview
2. Model Comparison
3. Texture Diagnostics
4. Diffusion Uncertainty
5. Case Reports
6. Reports

### B. Case report index

Path:

- `{to_project_relative_path(input_paths["case_report_index"])}`

Purpose:

- show selected individual examples,
- compare clean/damaged/mask/model outputs,
- inspect uncertainty and texture diagnostics where available.

### C. Stable Diffusion uncertainty heatmap report

Path:

- `{to_project_relative_path(input_paths["uncertainty_heatmap_report"])}`

Purpose:

- show seed-based spatial instability,
- explain masked, bounding-box, outside-mask, and boundary-ring uncertainty.

### D. Refined comparison output

Path:

- `{to_project_relative_path(input_paths["refined_comparison"])}`

Purpose:

- show final reference-based model comparison under the refined metric-region policy.

## 3. Meeting goals

The meeting should answer these questions:

1. Is the 50-painting controlled subset sufficient?
2. Is the refined metric-region policy accepted?
3. Should texture and brushstroke-proxy diagnostics remain part of the core framework?
4. Is the 40-case Stable Diffusion uncertainty subset sufficient for the thesis?
5. Should uncertainty heatmaps be expanded to all 200 non-zero cases?
6. Should SDXL remain feasibility-audited only?
7. Should semantic/iconographic checks be added after feedback?
8. Should metadata-driven analysis be added after feedback?
9. Should metric-policy ablation be added after feedback?
10. Should the Streamlit dashboard be treated as a formal supporting artifact?

## 4. Recommended presentation order

### Step 1 — Thesis framing

Message:

> The thesis is not about training a better restoration model. It is about evaluating restoration trustworthiness.

Core claim:

> Visual plausibility is not the same as restoration trustworthiness.

### Step 2 — Controlled benchmark

Show:

- 50 paintings,
- 5 categories,
- 200 non-zero cases,
- 3 evaluated models,
- SDXL feasibility audit.

### Step 3 — Refined model comparison

Show:

- LaMa dominance under refined full-reference metrics,
- OpenCV as deterministic baseline,
- Stable Diffusion weak under reference metrics but important diagnostically.

### Step 4 — Metric-region policy

Explain:

- MSE/PSNR on masked region,
- SSIM/LPIPS/CLIP/DINOv2 on mask bounding-box crop,
- texture and brushstroke-proxy also on mask bounding-box crop.

### Step 5 — Texture diagnostics

Explain:

- added local texture layer,
- brushstroke-proxy is a directional texture proxy,
- not semantic recognition or authentication.

### Step 6 — Uncertainty heatmaps

Explain:

- 40 selected cases,
- 4 seeds per case,
- spatial variation across Stable Diffusion outputs,
- not calibrated confidence.

### Step 7 — Case reports

Show:

- selected examples from `case_report_index.html`,
- cases where metrics, texture, and uncertainty reveal different behavior.

### Step 8 — Ask for scope decisions

Ask supervisor to approve, reject, or prioritize:

- scaling beyond 50 paintings,
- expanding uncertainty to all 200 non-zero cases,
- SDXL follow-up,
- semantic/iconographic checks,
- metadata-driven analysis,
- metric-policy ablation,
- human/expert review.

## 5. Recommended pre-feedback stance

Do not start major new experiments before supervisor feedback.

The current package is sufficient to ask for methodological approval and scope direction.
"""

write_text(
    output_paths["supervisor_feedback_agenda"],
    supervisor_feedback_agenda_md,
)

print("Saved:", output_paths["supervisor_feedback_agenda"])
print(output_paths["supervisor_feedback_agenda"].read_text(encoding="utf-8")[:3000])

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_feedback_agenda.md
# Supervisor Feedback Agenda

This agenda is intended for the next supervisor meeting.

## 1. One-sentence project status

The project now has a working pre-feedback evaluation framework for AI-assisted painting restoration, including refined full-reference metrics, texture and brushstroke-proxy diagnostics, Stable Diffusion uncertainty heatmaps, selected per-case diagnostic reports, and an updated Streamlit dashboard.

## 2. Main artifacts to show

### A. Streamlit dashboard

Run command:

    streamlit run streamlit_app.py

Recommended pages to show:

1. Overview
2. Model Comparison
3. Texture Diagnostics
4. Diffusion Uncertainty
5. Case Reports
6. Reports

### B. Case report index

Path:

- `outputs/reports/case_diagnostics/case_report_index.html`

Purpose:

- show selected individual examples,
- compare clean/damaged/mask/model outputs,
- inspect uncertainty and texture di

In [13]:
artifact_rows = [
    {
        "artifact_group": "notebook",
        "artifact_name": "31_texture_metrics_cleaned.ipynb",
        "path": "notebooks/31_texture_metrics_cleaned.ipynb",
        "status": "completed",
        "purpose": "Texture and brushstroke-proxy diagnostics.",
        "show_to_supervisor": True,
    },
    {
        "artifact_group": "notebook",
        "artifact_name": "32_uncertainty_heatmaps_cleaned.ipynb",
        "path": "notebooks/32_uncertainty_heatmaps_cleaned.ipynb",
        "status": "completed",
        "purpose": "Stable Diffusion spatial uncertainty heatmaps.",
        "show_to_supervisor": True,
    },
    {
        "artifact_group": "notebook",
        "artifact_name": "33_case_report_generation_cleaned.ipynb",
        "path": "notebooks/33_case_report_generation_cleaned.ipynb",
        "status": "completed",
        "purpose": "Selected per-case diagnostic reports.",
        "show_to_supervisor": True,
    },
    {
        "artifact_group": "notebook",
        "artifact_name": "34_prepare_final_dashboard_assets_cleaned.ipynb",
        "path": "notebooks/34_prepare_final_dashboard_assets_cleaned.ipynb",
        "status": "completed",
        "purpose": "Dashboard-ready CSV/JSON assets.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "notebook",
        "artifact_name": "35_refresh_supervisor_package_cleaned.ipynb",
        "path": "notebooks/35_refresh_supervisor_package_cleaned.ipynb",
        "status": "completed",
        "purpose": "Refresh supervisor package outputs.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "dashboard",
        "artifact_name": "streamlit_app.py",
        "path": "streamlit_app.py",
        "status": "completed",
        "purpose": "Interactive dashboard for reviewing final pre-feedback framework.",
        "show_to_supervisor": True,
    },
    {
        "artifact_group": "dashboard",
        "artifact_name": "dashboard_summary.json",
        "path": to_project_relative_path(input_paths["dashboard_summary"]),
        "status": "completed",
        "purpose": "Dashboard overview facts.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "dashboard",
        "artifact_name": "dashboard_asset_manifest.json",
        "path": to_project_relative_path(input_paths["dashboard_asset_manifest"]),
        "status": "completed",
        "purpose": "Manifest of dashboard-ready assets.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "report",
        "artifact_name": "case_report_index.html",
        "path": to_project_relative_path(input_paths["case_report_index"]),
        "status": "completed",
        "purpose": "Main entry point for selected per-case diagnostic reports.",
        "show_to_supervisor": True,
    },
    {
        "artifact_group": "report",
        "artifact_name": "stable_diffusion_uncertainty_heatmap_report_50.html",
        "path": to_project_relative_path(input_paths["uncertainty_heatmap_report"]),
        "status": "completed",
        "purpose": "Stable Diffusion spatial uncertainty heatmap report.",
        "show_to_supervisor": True,
    },
    {
        "artifact_group": "metrics",
        "artifact_name": "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
        "path": to_project_relative_path(input_paths["refined_comparison"]),
        "status": "completed",
        "purpose": "Final refined comparison across OpenCV Telea, LaMa, and Stable Diffusion.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "metrics",
        "artifact_name": "comparison_texture_unified_50.csv",
        "path": to_project_relative_path(input_paths["texture_unified"]),
        "status": "completed" if input_paths["texture_unified"].exists() else "missing",
        "purpose": "Unified texture and brushstroke-proxy diagnostics.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "metrics",
        "artifact_name": "stable_diffusion_uncertainty_heatmap_summary_by_case_50.csv",
        "path": to_project_relative_path(input_paths["uncertainty_heatmap_summary_by_case"]),
        "status": "completed" if input_paths["uncertainty_heatmap_summary_by_case"].exists() else "missing",
        "purpose": "Case-level uncertainty heatmap summaries.",
        "show_to_supervisor": False,
    },
    {
        "artifact_group": "metrics",
        "artifact_name": "case_diagnostic_report_manifest_50.csv",
        "path": to_project_relative_path(input_paths["case_diagnostic_report_manifest"]),
        "status": "completed",
        "purpose": "Manifest of selected case reports and diagnostic grids.",
        "show_to_supervisor": False,
    },
]

supervisor_artifact_index_df = pd.DataFrame(artifact_rows)

supervisor_artifact_index_df["exists"] = supervisor_artifact_index_df["path"].apply(file_exists_from_value)

supervisor_artifact_index_df.to_csv(
    output_paths["supervisor_artifact_index"],
    index=False,
)

print("Saved:", output_paths["supervisor_artifact_index"])

display(supervisor_artifact_index_df)

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_artifact_index.csv


,artifact_group,artifact_name,path,status,purpose,show_to_supervisor,exists
0,notebook,31_texture_metrics_cleaned.ipynb,notebooks/31_texture_metrics_cleaned.ipynb,completed,Texture and brushstroke-proxy diagnostics.,True,True
1,notebook,32_uncertainty_heatmaps_cleaned.ipynb,notebooks/32_uncertainty_heatmaps_cleaned.ipynb,completed,Stable Diffusion spatial uncertainty heatmaps.,True,True
2,notebook,33_case_report_generation_cleaned.ipynb,notebooks/33_case_report_generation_cleaned.ipynb,completed,Selected per-case diagnostic reports.,True,True
3,notebook,34_prepare_final_dashboard_assets_cleaned.ipynb,notebooks/34_prepare_final_dashboard_assets_cl...,completed,Dashboard-ready CSV/JSON assets.,False,True
4,notebook,35_refresh_supervisor_package_cleaned.ipynb,notebooks/35_refresh_supervisor_package_cleane...,completed,Refresh supervisor package outputs.,False,True
5,dashboard,streamlit_app.py,streamlit_app.py,completed,Interactive dashboard for reviewing final pre-...,True,True
6,dashboard,dashboard_summary.json,outputs/dashboard/dashboard_summary.json,completed,Dashboard overview facts.,False,True
7,dashboard,dashboard_asset_manifest.json,outputs/dashboard/dashboard_asset_manifest.json,completed,Manifest of dashboard-ready assets.,False,True
8,report,case_report_index.html,outputs/reports/case_diagnostics/case_report_i...,completed,Main entry point for selected per-case diagnos...,True,True
9,report,stable_diffusion_uncertainty_heatmap_report_50...,outputs/reports/stable_diffusion_uncertainty_h...,completed,Stable Diffusion spatial uncertainty heatmap r...,True,True


In [14]:
supervisor_package_manifest = {
    "generated_by": NOTEBOOK_NAME,
    "generated_at": supervisor_facts["generated_at"],
    "package_directory": to_project_relative_path(supervisor_package_dir),
    "refresh_policy": {
        "versioning": "Existing supervisor package outputs are refreshed in place.",
        "no_duplicate_v2_files": True,
    },
    "outputs": {
        key: {
            "path": to_project_relative_path(path),
            "exists": path.exists(),
            "file_size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        }
        for key, path in output_paths.items()
    },
    "source_artifacts": {
        key: {
            "path": to_project_relative_path(path),
            "exists": path.exists(),
        }
        for key, path in input_paths.items()
    },
    "supervisor_facts": supervisor_facts,
    "ready_for_supervisor_feedback": True,
    "recommended_next_action": (
        "Review Streamlit dashboard, selected case reports, and open questions with supervisor before "
        "starting scope-expanding experiments."
    ),
}

write_json(
    output_paths["supervisor_package_manifest"],
    supervisor_package_manifest,
)

print("Saved:", output_paths["supervisor_package_manifest"])
print(json.dumps(supervisor_package_manifest, indent=2, ensure_ascii=False)[:4000])

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_package_manifest.json
{
  "generated_by": "35_refresh_supervisor_package_cleaned",
  "generated_at": "2026-07-07T22:29:38",
  "package_directory": "outputs/supervisor_package",
  "refresh_policy": {
    "versioning": "Existing supervisor package outputs are refreshed in place.",
    "no_duplicate_v2_files": true
  },
  "outputs": {
    "supervisor_summary": {
      "path": "outputs/supervisor_package/supervisor_summary.md",
      "exists": true,
      "file_size_kb": 6.29
    },
    "supervisor_artifact_index": {
      "path": "outputs/supervisor_package/supervisor_artifact_index.csv",
      "exists": true,
      "file_size_kb": 2.39
    },
    "supervisor_key_findings": {
      "path": "outputs/supervisor_package/supervisor_key_findings.json",
      "exists": true,
      "file_size_kb": 5.06
    },
    "supervisor_open_questions": {
      "path": "outputs/supervisor_package/supervisor_open_ques

In [15]:
required_supervisor_outputs = [
    output_paths["supervisor_summary"],
    output_paths["supervisor_artifact_index"],
    output_paths["supervisor_key_findings"],
    output_paths["supervisor_open_questions"],
    output_paths["supervisor_feedback_agenda"],
    output_paths["supervisor_package_manifest"],
]

missing_outputs = [
    path
    for path in required_supervisor_outputs
    if not path.exists()
]

if missing_outputs:
    for path in missing_outputs:
        print("Missing:", path)

    raise FileNotFoundError("Some required supervisor package outputs are missing.")

artifact_index_check_df = pd.read_csv(output_paths["supervisor_artifact_index"])

if artifact_index_check_df.empty:
    raise ValueError("Supervisor artifact index is empty.")

if not artifact_index_check_df["exists"].all():
    missing_artifacts_df = artifact_index_check_df[~artifact_index_check_df["exists"]]
    display(missing_artifacts_df)
    raise FileNotFoundError("Some supervisor artifact index entries do not exist.")

manifest_payload = load_json_if_exists(output_paths["supervisor_package_manifest"])

if not manifest_payload:
    raise ValueError("Supervisor package manifest JSON is empty.")

if not manifest_payload.get("ready_for_supervisor_feedback", False):
    raise ValueError("Supervisor package manifest is not marked feedback-ready.")

summary_text = output_paths["supervisor_summary"].read_text(encoding="utf-8")

required_summary_phrases = [
    "Texture and brushstroke-proxy diagnostics",
    "Stable Diffusion uncertainty heatmaps",
    "Per-case diagnostic reports",
    "Final dashboard assets and updated Streamlit app",
    "Remaining work after supervisor feedback",
]

missing_summary_phrases = [
    phrase
    for phrase in required_summary_phrases
    if phrase not in summary_text
]

if missing_summary_phrases:
    raise ValueError(f"Supervisor summary missing expected phrases: {missing_summary_phrases}")

open_questions_text = output_paths["supervisor_open_questions"].read_text(encoding="utf-8")

required_open_question_phrases = [
    "uncertainty be expanded to all 200 non-zero cases",
    "SDXL remain feasibility-audited only",
    "Semantic or iconographic consistency checks",
    "Metadata-driven analysis",
    "Metric-policy ablation",
]

missing_open_question_phrases = [
    phrase
    for phrase in required_open_question_phrases
    if phrase not in open_questions_text
]

if missing_open_question_phrases:
    raise ValueError(f"Open questions missing expected phrases: {missing_open_question_phrases}")

print("Notebook 35 final validation passed.")
print("Supervisor package directory:", supervisor_package_dir)

print("\nSupervisor package outputs:")
for path in required_supervisor_outputs:
    print("-", path, "|", round(path.stat().st_size / 1024, 2), "KB")

print("\nShow-to-supervisor artifacts:")
display(
    artifact_index_check_df[
        artifact_index_check_df["show_to_supervisor"] == True
    ][
        [
            "artifact_group",
            "artifact_name",
            "path",
            "purpose",
        ]
    ]
)

Notebook 35 final validation passed.
Supervisor package directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package

Supervisor package outputs:
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_summary.md | 6.29 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_artifact_index.csv | 2.39 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_key_findings.json | 5.06 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_open_questions.md | 5.16 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_feedback_agenda.md | 4.01 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\supervisor_package\supervisor_package_manifest.json | 9.11 KB

Show-to-supervisor artifacts:


,artifact_group,artifact_name,path,purpose
0,notebook,31_texture_metrics_cleaned.ipynb,notebooks/31_texture_metrics_cleaned.ipynb,Texture and brushstroke-proxy diagnostics.
1,notebook,32_uncertainty_heatmaps_cleaned.ipynb,notebooks/32_uncertainty_heatmaps_cleaned.ipynb,Stable Diffusion spatial uncertainty heatmaps.
2,notebook,33_case_report_generation_cleaned.ipynb,notebooks/33_case_report_generation_cleaned.ipynb,Selected per-case diagnostic reports.
5,dashboard,streamlit_app.py,streamlit_app.py,Interactive dashboard for reviewing final pre-...
8,report,case_report_index.html,outputs/reports/case_diagnostics/case_report_i...,Main entry point for selected per-case diagnos...
9,report,stable_diffusion_uncertainty_heatmap_report_50...,outputs/reports/stable_diffusion_uncertainty_h...,Stable Diffusion spatial uncertainty heatmap r...


## Notebook 35 summary

This notebook refreshed the supervisor-facing package in place.

It updated the existing supervisor package outputs instead of creating duplicate versioned files.

Main refreshed outputs:

- `outputs/supervisor_package/supervisor_summary.md`
- `outputs/supervisor_package/supervisor_artifact_index.csv`
- `outputs/supervisor_package/supervisor_key_findings.json`
- `outputs/supervisor_package/supervisor_open_questions.md`
- `outputs/supervisor_package/supervisor_feedback_agenda.md`
- `outputs/supervisor_package/supervisor_package_manifest.json`

The refreshed package documents what was added since the original supervisor package:

- texture and brushstroke-proxy diagnostics,
- Stable Diffusion uncertainty heatmaps,
- selected per-case diagnostic reports,
- final dashboard assets,
- updated Streamlit app.

It also separates work that is ready for supervisor review from work that should wait until after feedback:

- semantic/iconographic consistency checks,
- metadata-driven analysis,
- metric-policy ablation,
- scaling beyond the controlled 50-painting subset,
- expanding uncertainty heatmaps to all 200 non-zero cases,
- SDXL follow-up,
- possible human or expert review,
- thesis chapter and paper structuring.

The package is now ready for supervisor feedback.